# 02 Exploratory Data Analysis (Stage 3)

**Training set only.** The test set stays untouched until Stage 12 (decision A6).

- **Part A:** data quality and leakage checks
- **Part B:** what is linked to churn (charts)
- **Part C:** overlapping columns and business insights

Every result is a count or a rate. We never print single customers.

## Setup

In [ ]:
# Tools: pandas for tables, and our own package for settings and loading.
import pandas as pd

from churn.config import POSITIVE_LABEL, TARGET_COLUMN
from churn.data import load_train_data

PERCENT = 100  # turns a share (0.288) into a percentage (28.8)
UNKNOWN_LABEL = "Unknown"  # the word the data uses for hidden missing values

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

# Load the TRAINING set only (the shared loader also runs our 3 checks).
train_df = load_train_data()

# True/False per customer: did they churn? Makes rates easy (mean = share True).
is_churn = train_df[TARGET_COLUMN].eq(POSITIVE_LABEL)
overall_churn_pct = is_churn.mean() * PERCENT

print(f"Training rows: {len(train_df):,} | columns: {train_df.shape[1]}")
print(f"Overall churn rate: {overall_churn_pct:.1f}%")

In [ ]:
# Helper used by every check below, so the logic is written once.
# Give it any grouping (e.g. "price unknown: True/False") and it returns,
# per group: number of customers, churn rate %, and the gap from the overall rate.
def churn_rate_by(groups: pd.Series, churned: pd.Series) -> pd.DataFrame:
    """Customers, churn rate (%) and gap from overall churn, per group.

    Args:
        groups: The group of each customer (same rows as churned).
        churned: True if the customer churned.

    Returns:
        One row per group: customers, churn_pct, gap_vs_overall_pct.
    """
    # dropna=False keeps missing values as their own group.
    summary = churned.groupby(groups, dropna=False).agg(
        customers="size", churn_pct="mean"
    )
    summary["churn_pct"] = summary["churn_pct"].mul(PERCENT).round(1)
    # Positive gap = this group churns MORE than average (in % points).
    overall_pct = churned.mean() * PERCENT
    summary["gap_vs_overall_pct"] = (summary["churn_pct"] - overall_pct).round(1)
    return summary

## A1. Missing values

Questions:
1. Are the customers missing the 6 usage columns the **same** customers each time?
2. Do customers with missing values churn more or less than average?

In [ ]:
# The 6 usage columns that were missing together in Stage 2.
USAGE_COLUMNS = [
    "MonthlyRevenue",
    "MonthlyMinutes",
    "TotalRecurringCharge",
    "DirectorAssistedCalls",
    "OverageMinutes",
    "RoamingCalls",
]

# isna() gives True per empty cell. any(axis=1) = row missing AT LEAST one,
# all(axis=1) = row missing ALL six. If both counts match, it is the same rows.
usage_missing = train_df[USAGE_COLUMNS].isna()
missing_any_usage = usage_missing.any(axis=1)
missing_all_usage = usage_missing.all(axis=1)
print(f"Rows missing at least one usage column: {missing_any_usage.sum():,}")
print(f"Rows missing all six usage columns:     {missing_all_usage.sum():,}")

# Are customers with missing usage also missing the % change columns?
missing_perc_change = train_df["PercChangeMinutes"].isna()
both_missing = (missing_all_usage & missing_perc_change).sum()
print(f"Missing % change columns: {missing_perc_change.sum():,}")
print(f"...of which also missing all usage columns: {both_missing:,}")

In [ ]:
# Churn rate for customers WITH vs WITHOUT each kind of missing value.
# True = value missing. A big gap means "missing" itself carries information.
missing_flags = {
    "usage columns missing": missing_all_usage,
    "% change columns missing": missing_perc_change,
    "AgeHH1 missing": train_df["AgeHH1"].isna(),
    "ServiceArea missing": train_df["ServiceArea"].isna(),
}

# pd.concat stacks the small tables into one, labelled by check name.
pd.concat(
    {name: churn_rate_by(flag, is_churn) for name, flag in missing_flags.items()},
    names=["check", "value_missing"],
)

## A2. Hidden missing values ("Unknown" and 0)

Questions:
1. Do customers with "Unknown" (or age/income 0) churn differently?
2. Are age 0 and income 0 the same customers (one "no demographic data" group)?

In [ ]:
# True = the value is a hidden "unknown".
hidden_unknown_flags = {
    "HandsetPrice = Unknown": train_df["HandsetPrice"].eq(UNKNOWN_LABEL),
    "MaritalStatus = Unknown": train_df["MaritalStatus"].eq(UNKNOWN_LABEL),
    "Homeownership = Unknown": train_df["Homeownership"].eq(UNKNOWN_LABEL),
    "AgeHH1 = 0": train_df["AgeHH1"].eq(0),
    "IncomeGroup = 0": train_df["IncomeGroup"].eq(0),
}

pd.concat(
    {
        name: churn_rate_by(flag, is_churn)
        for name, flag in hidden_unknown_flags.items()
    },
    names=["check", "is_unknown"],
)

In [ ]:
# Crosstab = a count table of two True/False columns against each other.
# If most customers sit on the diagonal (True/True, False/False),
# age 0 and income 0 mostly happen together.
pd.crosstab(
    hidden_unknown_flags["AgeHH1 = 0"],
    hidden_unknown_flags["IncomeGroup = 0"],
    rownames=["AgeHH1 = 0"],
    colnames=["IncomeGroup = 0"],
)

## A3. Retention columns: leakage check (decision A2)

Question: do these columns give away the answer?

Warning signs of leakage:
- A group with churn close to **100%** (or 0%)
- A link much stronger than any normal behaviour column

A strong link alone does not prove leakage. The real question is **timing**: was the call made before or after the snapshot?

In [ ]:
# The 3 columns flagged in decision A2.
RETENTION_COLUMNS = [
    "MadeCallToRetentionTeam",
    "RetentionCalls",
    "RetentionOffersAccepted",
]

pd.concat(
    {column: churn_rate_by(train_df[column], is_churn) for column in RETENTION_COLUMNS},
    names=["column", "value"],
)

## A4. Odd values

Questions:
1. How many customers have impossible negative values?
2. What do the very large accounts (`UniqueSubs`, `ActiveSubs`) look like?

We only look here. Fixes are decided in Stage 4 (Cleaning).

In [ ]:
# Columns that should never be negative (money and days).
# PercChange columns are left out: a % change CAN be negative.
NON_NEGATIVE_COLUMNS = [
    "MonthlyRevenue",
    "TotalRecurringCharge",
    "CurrentEquipmentDays",
]

negative_counts = pd.Series(
    {column: (train_df[column] < 0).sum() for column in NON_NEGATIVE_COLUMNS},
    name="customers_with_negative_value",
)
negative_counts

In [ ]:
# Spread of subscriber counts. High percentiles show how rare big accounts are.
# 0.99 = 99% of customers are at or below this value.
SUBSCRIBER_PERCENTILES = [0.5, 0.9, 0.99, 0.999]
train_df[["UniqueSubs", "ActiveSubs"]].describe(percentiles=SUBSCRIBER_PERCENTILES).T

In [ ]:
# Do big accounts churn differently? And a logic check:
# active subscribers should never be more than unique (total) subscribers.
LARGE_ACCOUNT_SUBS = 5  # more than this many subscribers = "large account"

is_large_account = train_df["UniqueSubs"].gt(LARGE_ACCOUNT_SUBS)
active_above_unique = train_df["ActiveSubs"].gt(train_df["UniqueSubs"]).sum()
print(f"Customers with ActiveSubs > UniqueSubs: {active_above_unique:,}\n")

churn_rate_by(is_large_account.rename("large_account"), is_churn)

# Part B: What is linked to churn?

Same idea as Part A: compare each group's churn rate with the average (28.8%).

**How to read the charts**
- Each bar = one group of customers. Its height = the % of that group who churned.
- The thin dark line = the average churn rate (28.8%). Bars above it churn more than average.
- **Grey bars** = fewer than 100 customers: too small to trust.
- Only the highest and lowest bars are labelled. The table under each chart has every number.

**Number columns** are split into 10 groups of about equal size ("deciles"), from lowest to highest value. If the bars go clearly up or down from left to right, the column is linked to churn.

In [ ]:
# Chart tools: matplotlib draws the charts, PercentFormatter shows "30%" on the axis.
import math

import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter

# --- Chart style: colours from the dataviz skill's reference palette (light mode) ---
SURFACE_COLOR = "#fcfcfb"  # chart background
BAR_COLOR = "#2a78d6"  # blue: one colour for every reliable bar
SMALL_GROUP_COLOR = "#c3c2b7"  # grey: groups too small to trust
TEXT_PRIMARY = "#0b0b0b"  # titles and value labels
TEXT_SECONDARY = "#52514e"  # average line and group labels
TEXT_MUTED = "#898781"  # axis numbers
GRID_COLOR = "#e1e0d9"  # thin background lines
AXIS_COLOR = "#c3c2b7"  # bottom axis line

N_BINS = 10  # number columns are split into 10 equal-size groups
MIN_GROUP_SIZE = 100  # groups smaller than this are drawn grey (unreliable)
BAR_WIDTH = 0.6  # bar thickness in the busiest chart (1.0 would make bars touch)
REFERENCE_BAR_COUNT = N_BINS + 1  # busiest chart: 10 groups + "Missing"
MIN_BAR_SLOTS = 4  # small charts get at least 4 slots, so bars sit centred
CHART_COLUMNS = 2  # charts per row in a grid
CHART_ROW_HEIGHT = 3.4  # height of each row of charts (inches)
CHART_WIDTH = 11  # width of the whole grid (inches)
Y_HEADROOM = 1.15  # 15% space above the tallest bar for its label
MISSING_LABEL = "Missing"  # group name for empty cells


# Show a number briefly: whole numbers without decimals, big numbers with commas.
def format_number(value: float) -> str:
    """Format a bin edge for a chart label, e.g. 6, 1,234 or 0.7."""
    if float(value).is_integer() or abs(value) >= PERCENT:
        return f"{value:,.0f}"
    return f"{value:.1f}"


# Split a number column into up to 10 groups with about the same number of
# customers in each. Labels show the real lowest and highest value in each group.
def make_bins(values: pd.Series, n_bins: int = N_BINS) -> pd.Series:
    """Group a number column into equal-size bins with readable labels.

    Args:
        values: The number column to split.
        n_bins: The number of groups wanted. Fewer are made when many
            customers share the same value (for example, lots of 0s).

    Returns:
        The group of each customer, in order from lowest to highest value.
        Empty cells get their own "Missing" group.
    """
    # qcut = "quantile cut": each group gets about the same number of customers.
    # duplicates="drop" merges groups whose edges are equal (e.g. many zeros).
    binned = pd.qcut(values, q=n_bins, duplicates="drop")

    # Label each group with the real smallest and largest value inside it.
    # "to" instead of a dash keeps negative ranges readable ("-3,406 to -229").
    ranges = values.groupby(binned).agg(["min", "max"])
    labels = [
        format_number(low)
        if low == high
        else f"{format_number(low)} to {format_number(high)}"
        for low, high in ranges.itertuples(index=False)
    ]
    binned = binned.cat.rename_categories(labels)

    if values.isna().any():
        binned = binned.cat.add_categories(MISSING_LABEL).fillna(MISSING_LABEL)
    return binned


# Draw ONE chart: churn rate per group, with the average line.
def plot_churn_rate(summary: pd.DataFrame, title: str, ax: plt.Axes) -> None:
    """Draw churn rate per group as bars on the given chart area.

    Args:
        summary: Output of churn_rate_by (customers, churn_pct per group).
        title: Chart title (the column name).
        ax: The chart area to draw on.
    """
    positions = list(range(len(summary)))
    is_reliable = summary["customers"] >= MIN_GROUP_SIZE
    colors = [BAR_COLOR if reliable else SMALL_GROUP_COLOR for reliable in is_reliable]

    # Same bar thickness in every chart, and bars centred in small charts.
    # slots = how many bar-widths of space the chart has (at least 4).
    slots = max(len(summary), MIN_BAR_SLOTS)
    side_space = (slots - len(summary)) / 2
    ax.set_xlim(-0.5 - side_space, len(summary) - 0.5 + side_space)
    bar_width = BAR_WIDTH * slots / REFERENCE_BAR_COUNT
    ax.bar(
        positions, summary["churn_pct"], width=min(bar_width, BAR_WIDTH), color=colors
    )
    ax.axhline(overall_churn_pct, color=TEXT_SECONDARY, linewidth=1)

    # Label only the highest and lowest reliable bars (not every bar).
    reliable_rates = summary.loc[is_reliable, "churn_pct"]
    if not reliable_rates.empty:
        for group in {reliable_rates.idxmax(), reliable_rates.idxmin()}:
            position = summary.index.get_loc(group)
            rate = summary.loc[group, "churn_pct"]
            ax.annotate(
                f"{rate:.0f}%",
                (position, rate),
                xytext=(0, 3),
                textcoords="offset points",
                ha="center",
                va="bottom",
                fontsize=8,
                color=TEXT_PRIMARY,
            )

    # Quiet styling: thin grid, no box around the chart, muted axis text.
    ax.set_title(title, loc="left", fontsize=10, color=TEXT_PRIMARY)
    ax.set_xticks(positions, summary.index.astype(str), rotation=45, ha="right")
    ax.tick_params(axis="x", length=0, labelsize=7, colors=TEXT_SECONDARY)
    ax.tick_params(axis="y", length=0, labelsize=8, colors=TEXT_MUTED)
    ax.yaxis.set_major_formatter(PercentFormatter(decimals=0))
    ax.grid(axis="y", color=GRID_COLOR, linewidth=0.8)
    ax.set_axisbelow(True)
    for side in ("top", "right", "left"):
        ax.spines[side].set_visible(False)
    ax.spines["bottom"].set_color(AXIS_COLOR)
    ax.set_facecolor(SURFACE_COLOR)


# Draw a GRID of charts that share the same y-axis, so they compare fairly.
def plot_churn_grid(summaries: dict[str, pd.DataFrame], title: str) -> None:
    """Draw one churn-rate chart per column in a grid with a shared y-axis.

    Args:
        summaries: Column name -> output of churn_rate_by.
        title: Title for the whole grid.
    """
    n_rows = math.ceil(len(summaries) / CHART_COLUMNS)
    fig, axes = plt.subplots(
        n_rows,
        CHART_COLUMNS,
        figsize=(CHART_WIDTH, CHART_ROW_HEIGHT * n_rows),
        sharey=True,
        squeeze=False,
        layout="constrained",
        facecolor=SURFACE_COLOR,
    )

    for ax, (name, summary) in zip(axes.flat, summaries.items(), strict=False):
        plot_churn_rate(summary, name, ax)
    # Hide empty chart areas when the number of charts is odd.
    for ax in axes.flat[len(summaries) :]:
        ax.set_visible(False)

    tallest_bar = max(summary["churn_pct"].max() for summary in summaries.values())
    axes[0, 0].set_ylim(0, tallest_bar * Y_HEADROOM)
    fig.suptitle(
        f"{title}\nLine = average churn ({overall_churn_pct:.1f}%). "
        f"Grey = fewer than {MIN_GROUP_SIZE} customers.",
        x=0.01,
        ha="left",
        fontsize=11,
        color=TEXT_PRIMARY,
    )
    plt.show()

## B1. Number columns

Questions:
- Do **newer** or **longer-standing** customers churn more? (`MonthsInService`)
- Do customers with **older phones** churn more? (`CurrentEquipmentDays`, the brief's example)
- Does churn change with **usage**, **bill size**, **customer care calls** or **dropped calls**?
- Do customers whose usage is **falling** churn more? (`PercChangeMinutes`)

In [ ]:
# The number columns to explore, chosen for the questions above.
NUMBER_COLUMNS = [
    "MonthsInService",
    "CurrentEquipmentDays",
    "MonthlyMinutes",
    "PercChangeMinutes",
    "MonthlyRevenue",
    "OverageMinutes",
    "CustomerCareCalls",
    "DroppedCalls",
]

# For each column: split into 10 groups, then churn rate per group.
number_summaries = {
    column: churn_rate_by(make_bins(train_df[column]), is_churn)
    for column in NUMBER_COLUMNS
}

plot_churn_grid(number_summaries, "Churn rate by number column (lowest to highest)")

In [ ]:
# Table version of the charts above: every number, for checking and for the report.
pd.concat(number_summaries, names=["column", "group"])

## B2. Category columns

Questions:
- Does churn change with **credit rating**?
- Does the **phone** matter: price, web-capable, refurbished?
- Does the **neighbourhood type** (`PrizmCode`) matter? (also part of the fairness check, decision A4)
- Do customers **new to mobile phones** churn differently?

In [ ]:
# The category columns to explore.
CATEGORY_COLUMNS = [
    "CreditRating",
    "HandsetPrice",
    "HandsetWebCapable",
    "HandsetRefurbished",
    "PrizmCode",
    "NewCellphoneUser",
]


# HandsetPrice is text ("10", "150", "Unknown"), so sorting it as text would put
# "150" before "30". This sorts it by price as a number, with "Unknown" last.
def handset_price_order(price: str) -> tuple[bool, float]:
    """Sort key for HandsetPrice: numbers low to high, then Unknown."""
    is_unknown = price == UNKNOWN_LABEL
    return is_unknown, 0.0 if is_unknown else float(price)


category_summaries = {
    column: churn_rate_by(train_df[column], is_churn) for column in CATEGORY_COLUMNS
}
# Put HandsetPrice groups in price order. CreditRating already sorts correctly
# because its labels start with a number ("1-Highest" ... "7-Lowest").
category_summaries["HandsetPrice"] = category_summaries["HandsetPrice"].loc[
    sorted(category_summaries["HandsetPrice"].index, key=handset_price_order)
]

plot_churn_grid(category_summaries, "Churn rate by category column")

In [ ]:
# Table version of the category charts.
pd.concat(category_summaries, names=["column", "group"])

## B3. Which columns matter most?

One simple score per column: **spread** = highest group churn minus lowest group churn (reliable groups only, 100+ customers).

- Big spread = churn changes a lot across the groups, so the column is strongly linked to churn.
- Small spread = churn is about the same in every group, so the column tells us little on its own.

Two versions:
- `spread_pts`: all groups, **including** "Missing".
- `spread_without_missing_pts`: real values only. For some columns the "Missing" group is the tallest bar, which makes the column look more important than its real values are. **This is the honest ranking.**

This is a rough first ranking, not proof. A column can also matter in combination with others, which the model will find later.

In [ ]:
# For each column: the spread of churn %, with and without the "Missing" group.
def churn_spread(summary: pd.DataFrame) -> pd.Series:
    """Spread of churn % across groups with 100+ customers, with and without Missing.

    Args:
        summary: Output of churn_rate_by for one column.

    Returns:
        spread_pts (all reliable groups) and spread_without_missing_pts.
    """
    reliable = summary[summary["customers"] >= MIN_GROUP_SIZE]
    all_rates = reliable["churn_pct"]
    # Real values only: drop the "Missing" group (if the column has one).
    value_rates = reliable.loc[reliable.index.astype(str) != MISSING_LABEL, "churn_pct"]
    return pd.Series(
        {
            "spread_pts": round(all_rates.max() - all_rates.min(), 1),
            "spread_without_missing_pts": round(
                value_rates.max() - value_rates.min(), 1
            ),
        }
    )


all_summaries = {**number_summaries, **category_summaries}
spread_table = pd.DataFrame(
    {column: churn_spread(summary) for column, summary in all_summaries.items()}
).T
# Honest ranking first: biggest spread of the real values at the top.
spread_table.sort_values("spread_without_missing_pts", ascending=False)

# Part C: Overlaps, areas and insights

- **C1.** Which columns carry the same information?
- **C2.** Does churn differ by service area (city)?
- **C3.** Are our two best findings (months in service, phone age) really separate?
- **C4.** Business insights for the report

## C1. Overlapping columns

Why it matters: two columns with the same information give the model nothing new. For logistic regression they also make the results unstable and the explanations confusing.

Questions:
1. Is `DroppedBlockedCalls` just `DroppedCalls + BlockedCalls`?
2. Which number columns move together almost perfectly?
3. Do the Yes/No pairs (`NewCellphoneUser`/`NotNewCellphoneUser`, retention columns) repeat each other?

In [ ]:
# Check 1: is DroppedBlockedCalls = DroppedCalls + BlockedCalls?
# Values are rounded averages, so we allow a tiny difference.
ROUNDING_TOLERANCE = 0.1

calls_sum = train_df["DroppedCalls"] + train_df["BlockedCalls"]
is_sum = (train_df["DroppedBlockedCalls"] - calls_sum).abs() <= ROUNDING_TOLERANCE
print(f"Rows where DroppedBlockedCalls = Dropped + Blocked: {is_sum.mean():.1%}")

In [ ]:
# Check 2: pairs of number columns that move together almost perfectly.
# Correlation: +1 = always rise together, 0 = unrelated,
# -1 = one rises as the other falls.
# Spearman correlation compares rankings, so extreme values cannot distort it.
import numpy as np

from churn.config import ID_COLUMN

HIGH_CORRELATION = 0.9  # pairs at or above this are "almost the same information"

numeric_df = train_df.select_dtypes("number").drop(columns=ID_COLUMN)
correlations = numeric_df.corr(method="spearman")

# Keep each pair once (the table is a mirror image), then keep only strong pairs.
upper_triangle = np.triu(np.ones(correlations.shape, dtype=bool), k=1)
pairs = (
    correlations.where(upper_triangle)
    .stack()
    .rename_axis(["column_a", "column_b"])
    .rename("correlation")
    .reset_index()
)
strong_pairs = pairs[pairs["correlation"].abs() >= HIGH_CORRELATION]
strong_pairs.sort_values("correlation", key=abs, ascending=False).round(2)

In [ ]:
# Check 3: do the Yes/No pairs repeat each other? (crosstab = count table)
retention_called = train_df["RetentionCalls"].gt(0).rename("RetentionCalls > 0")
display(pd.crosstab(retention_called, train_df["MadeCallToRetentionTeam"]))
display(pd.crosstab(train_df["NewCellphoneUser"], train_df["NotNewCellphoneUser"]))

## C2. Service areas

`ServiceArea` codes look like `NYCBRO917`. The first 3 letters seem to be the city or market (e.g. `NYC`).

Questions:
1. With 747 areas, how many have enough customers to trust their churn rate?
2. Does churn differ by city?

In [ ]:
# Churn rate per service area (747 areas). Most areas are small.
CITY_CODE_LENGTH = 3  # "NYCBRO917" -> "NYC"
TOP_BOTTOM_ROWS = 5  # cities to show at each end

area_summary = churn_rate_by(train_df["ServiceArea"], is_churn)
reliable_areas = area_summary[area_summary["customers"] >= MIN_GROUP_SIZE]
covered_share = reliable_areas["customers"].sum() / len(train_df)
print(f"Service areas: {len(area_summary):,}")
print(f"Areas with {MIN_GROUP_SIZE}+ customers: {len(reliable_areas):,}")
print(f"Customers in those areas: {covered_share:.0%}")

# Group areas by city code (first 3 letters): fewer, bigger groups.
city_codes = train_df["ServiceArea"].str[:CITY_CODE_LENGTH].rename("city_code")
city_summary = churn_rate_by(city_codes, is_churn)
reliable_cities = city_summary[city_summary["customers"] >= MIN_GROUP_SIZE]
reliable_cities = reliable_cities.sort_values("churn_pct")
print(f"City codes: {len(city_summary)}")
print(f"City codes with {MIN_GROUP_SIZE}+ customers: {len(reliable_cities)}")

# Lowest and highest churn cities.
pd.concat(
    {
        "lowest churn": reliable_cities.head(TOP_BOTTOM_ROWS),
        "highest churn": reliable_cities.tail(TOP_BOTTOM_ROWS),
    },
    names=["end", "city_code"],
)

## C3. Are months in service and phone age the same finding?

A customer who never changed phone has a phone about as old as their account. So "customers at 11-12 months churn more" and "older phones churn more" **could be the same customers**.

To separate them, we split customers into:
- **Same phone since joining:** phone age is about the same as months with the company.
- **Changed phone:** phone is clearly newer than the account.

Then we compare churn for each group, at each stage of the customer's life.

In [ ]:
# Phone age in months, to compare with months in service.
DAYS_PER_MONTH = 30.4
CHANGED_PHONE_GAP_MONTHS = 1  # phone at least 1 month newer than the account
TENURE_EDGES = [0, 10, 12, 18, float("inf")]  # month ranges: 6-10, 11-12, 13-18, 19+
TENURE_LABELS = ["6-10 months", "11-12 months", "13-18 months", "19+ months"]

phone_age_months = train_df["CurrentEquipmentDays"] / DAYS_PER_MONTH
changed_phone = phone_age_months < (
    train_df["MonthsInService"] - CHANGED_PHONE_GAP_MONTHS
)
tenure_band = pd.cut(train_df["MonthsInService"], TENURE_EDGES, labels=TENURE_LABELS)

# Churn per (months-in-service range, changed phone yes/no).
groups = [
    tenure_band.rename("months_in_service"),
    changed_phone.rename("changed_phone"),
]
phone_vs_tenure = is_churn.groupby(groups, observed=True).agg(
    customers="size", churn_pct="mean"
)
phone_vs_tenure["churn_pct"] = phone_vs_tenure["churn_pct"].mul(PERCENT).round(1)
# unstack = put "changed phone: False / True" side by side as columns.
phone_vs_tenure.unstack("changed_phone")

## C4. Business insights (for the manager)

Based on the **training set** (40,837 customers). Average churn: **28.8%**.

**1. The 1-year danger zone.**
Customers at **11-12 months** churn at **42.5%**, almost 3 times the rate at 6-8 months (15.9%). This happens whether or not they changed phone (42.8% vs 41.5%), so it is about timing, most likely the end of a 1-year contract.
*Action:* contact customers **before** month 11, around month 9-10.

**2. Long-standing customers on their original phone.**
After 19+ months, customers still on the phone they joined with churn at **33.9%**, vs **25.7%** for those who changed phone. Customers whose phone cannot use the internet churn at **37.2%** (vs 27.9%).
*Action:* phone upgrade offers for long-standing customers with old phones.

**3. Low or falling usage is an early warning.**
The lowest users (0-53 minutes a month) churn at **37.8%**, vs **25.0%** for the heaviest users. Customers with the biggest drop in minutes churn at **34.3%**. Customers with no recent usage data churn at **46-57%**.
*Action:* flag customers whose usage drops sharply.

**4. Customer care calls are not a churn signal on their own.**
Customers who call customer care the most churn **less** (24.5%) than customers who almost never call (30.5%). Targeting retention calls at "the customers who complain most" would miss many churners.

**5. Churn differs by city.**
Among cities with 100+ customers, churn ranges from **22.0%** (STL) to **41.7%** (SLC). Location is worth including in the model.

**Cautions**
- These are **links, not proven causes**. Business actions should be tested (e.g. offer to some customers, compare with others).
- The data is old and has no customers under 6 months, so these patterns may not hold for new customers or today's market.

## EDA summary: what this means for the next stages

| Finding | Proposed action | Stage |
|---|---|---|
| Missing usage data = strong churn signal | Fill gaps **and** add "was missing" flags | 7 (pipeline) |
| "Unknown" values = weak signal | Keep "Unknown" as its own category | 7 |
| A few impossible negatives (revenue, charges, phone days) | Rule-based fix (e.g. treat as missing) | 4 (cleaning) |
| `DroppedBlockedCalls` = Dropped + Blocked in 89% of rows | Likely redundant: consider dropping | 5 (features) |
| `Handsets` and `HandsetModels` move together (0.91) | Consider keeping only one | 5 |
| `MadeCallToRetentionTeam` = (`RetentionCalls` > 0) exactly | Duplicate. Retention columns still under decision A2 | 5 |
| `NewCellphoneUser` / `NotNewCellphoneUser` never both Yes | Combine into one 3-way column: new / not new / unknown | 4 |
| `ServiceArea`: 730 areas, most small | Use the 3-letter city code (58 cities) instead | 5/7 |
| `HandsetPrice` stored as text | Convert to number, with "Unknown" kept as information | 4 |
| Phone age vs time with company are separate effects | Keep both; consider a "changed phone" feature | 5 |
| Test set may contain values bigger than any in training (e.g. `UniqueSubs`) | Pipeline must handle unseen large values | 7 |

These are **proposals**. Each one is confirmed (or changed) at its stage, with the owner.